# NB6 — Descriptive & Clustering Visualisations

**Pipeline position:** runs after NB3 (clustering) and NB5 (regressions)
**Inputs:** `intermediary/Master.csv`, `intermediary/NaturalResource.csv`, `intermediary/clustersagg.csv`, `intermediary/clusters1995.csv`, `intermediary/sample_countries_final.csv`
**Outputs:** `main_analysis/Final/charts/descriptive/`

Charts produced:
- Chart 000: Sample country map
- Chart 02: ECI correlations by variable category
- Chart 03: PCA biplot (k=5, 1995)
- Chart 04: Cluster world map (k=5, 1995)
- Chart 05: Animated ECI vs log(GDP pc)
- Chart 06: Cluster profile comparison
- Chart 14: ECI distribution shift 1995 vs 2019
- Chart 15: ECI median trajectory by cluster
- Chart 26: PCA resource loadings heatmap


# Descriptive Statistics & Clustering Visualisations

**Capstone Project — Moody's Ratings**
*Industrial Upgrading in Emerging Markets with Rich Natural Resources*

Charts for Sections 3.2 (Data), 3.3 (Descriptive Statistics & Clusters), and Appendix. All data read from CSVs produced by NB0–NB5.

**Clustering: k=5**
Labels: Petrostates, Oil Exporters, Major Producers, Mining Exporters, Forestry Intensive


## 0. Setup

In [1]:
import os, sys, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

# Project root
ROOT = os.getcwd()
print(f"Project root: {ROOT}")

# Output directory
OUT = os.path.join('Final', 'charts', 'descriptive')
os.makedirs(OUT, exist_ok=True)

# Style constants
FONT = 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
WRITE_CONFIG = {'displayModeBar': False, 'responsive': True}

# Cluster colors (k=4, matching NB4 label assignment)
CLUSTER_COLORS = {
    'Petrostates':        '#E63946',
    'Oil Exporters':      '#457B9D',
    'Major Producers':    '#2A9D8F',
    'Mining Exporters':   '#E9C46A',
    'Forestry Intensive': '#8B5CF6',
}

# Helper functions
def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=60, r=40, t=40, b=50))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=2)
        print(f"  Saved: {path}.png")
    except Exception as e:
        print(f"  PNG skipped: {e}")

_FEAT_SHORT = {
    'Domestic credit to private sector (% of GDP)':                        'Domestic Credit',
    'Access to electricity (% of population)':                             'Electricity Access',
    'Human capital index':                                                  'Human Capital',
    'HCI_x_ProductionValue':                                               'HC x Production',
    'GFCF_x_ProductionValue':                                              'GFCF x Production',
    'Rule of law index':                                                    'Rule of Law',
    'Political stability \u2014 estimate':                                  'Political Stability',
    'Trade (% of GDP)':                                                     'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Capital Formation',
    'Urban population (% of total population)':                           'Urban Population',
    'Use of IMF credit (DOD, current US$)':                               'IMF Credit',
    'Total_Production_Value_Per_Capita':                                   'Prod Value p.c.',
    'Capital depreciation rate':                                           'Depreciation',
    'Landlocked':                                                          'Landlocked',
    'Real interest rate (%)':                                              'Interest Rate',
    'Inflation, consumer prices (annual %)':                               'Inflation',
    'GDP per capita (constant prices, PPP)':                               'GDP per Capita',
    'Total natural resources rents (% of GDP)':                           'NR Rents (% GDP)',
    'Oil rents (% of GDP)':                                                'Oil Rents',
    'Mineral rents (% of GDP)':                                            'Mineral Rents',
    'Natural gas rents (% of GDP)':                                        'Gas Rents',
    'Economic Complexity Index':                                           'ECI',
    'Adjusted savings: gross savings (% of GNI)':                        'Savings',
    'Government revenue':                                                   'Gov Revenue',
    'Share of investment in GDP':                                          'Investment Share',
    'Life expectancy at birth, total (years)':                            'Life Expectancy',
    'Death rate, crude (per 1,000 people)':                               'Death Rate',
    'Manufacturing, value added (% of GDP)':                              'Manufacturing',
    'Agriculture, forestry, and fishing, value added (% of GDP)':        'Agriculture (% GDP)',
    'Mobile cellular subscriptions (per 100 people)':                     'Mobile Subs',
    'Political corruption index':                                          'Pol. Corruption',
    'Property rights':                                                     'Property Rights',
    'Services, value added (% of GDP)':                                   'Services (% GDP)',
    'Industry (including construction), value added (% of GDP)':         'Industry (% GDP)',
    'prod_pc':                                                             'Prod Value p.c.',
}
def shorten(name, max_len=30):
    return _FEAT_SHORT.get(name, name[:max_len])

# Load data
master    = pd.read_csv('intermediary/Master.csv')
nr_full   = pd.read_csv('intermediary/NaturalResource.csv')
include   = pd.read_csv('intermediary/sample_countries_final.csv')['Country Code'].tolist()
clusters  = pd.read_csv('intermediary/clustersagg.csv')

panel = master[(master['Country Code'].isin(include)) &
               (master['Year'] >= 1995) & (master['Year'] <= 2019)].copy()
nr_sample = nr_full[nr_full['Country Code'].isin(include)]

# Attach cluster labels to panel
cl_map = clusters[['Country Code', 'Country', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
panel = panel.merge(cl_map, on='Country Code', how='left')

print(f"Panel: {panel['Country Code'].nunique()} countries, {len(panel):,} obs")
print(f"NR data: {nr_sample['Country Code'].nunique()} countries, {len(nr_sample):,} rows")
print(f"Clusters: {clusters['ClusterLabels'].value_counts().to_dict()}")

Project root: /Users/leoss/Desktop/GitHub/Capstone/Client/main_analysis
Panel: 54 countries, 1,350 obs
NR data: 54 countries, 6,480 rows
Clusters: {'Forestry Intensive': 21, 'Petrostates': 12, 'Oil Exporters': 11, 'Major Producers': 6, 'Mining Exporters': 4}


## Chart 000 — Sample countries

In [2]:
# Chart 000 — World map highlighting all countries in the sample
SAMPLE_COLOR = '#457B9D'

# Build a simple 0/1 dataframe: 1 = in sample
df_sample = (
    master[['Country Code', 'Country Name']].drop_duplicates('Country Code')
    .query('`Country Code` in @include')
    .assign(z=1)
)

fig000 = go.Figure()

fig000.add_trace(go.Choropleth(
    locations=df_sample['Country Code'],
    z=df_sample['z'],
    colorscale=[[0, SAMPLE_COLOR], [1, SAMPLE_COLOR]],
    showscale=False,
    showlegend=False,
    hovertemplate='<b>%{text}</b><extra></extra>',
    text=df_sample['Country Name'].values,
    marker=dict(line=dict(color='white', width=0.6)),
))

fig000.update_layout(
    **base_layout(
        title=dict(
            text='Sample countries (n={})'.format(len(df_sample)),
            x=0.5, xanchor='center',
            font=dict(size=15, color=NAVY),
        ),
        geo=dict(
            showframe=False,
            showcoastlines=False,
            showland=True,  landcolor='#f4f4f4',
            showocean=True, oceancolor='#ffffff',
            showlakes=False,
            bgcolor=BG,
            projection_type='natural earth',
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=460,
    )
)

fig000.show(config=WRITE_CONFIG)
save(fig000, 'chart000_sample_map', w=1200, h=500)

  Saved: Final/charts/descriptive/chart000_sample_map.png


## Chart 02 — Correlation of features with ECI (by category)

In [3]:
# Correlation bar chart colored by variable category (Figure 2 style)
CATEGORY_MAP = {
    'Total natural resources rents (% of GDP)': 'Resource Rents',
    'Oil rents (% of GDP)':                      'Resource Rents',
    'Mineral rents (% of GDP)':                  'Resource Rents',
    'Natural gas rents (% of GDP)':              'Resource Rents',
    'GDP per capita (constant prices, PPP)':     'Macro & Structure',
    'Manufacturing, value added (% of GDP)':     'Macro & Structure',
    'Agriculture, forestry, and fishing, value added (% of GDP)': 'Macro & Structure',
    'Services, value added (% of GDP)':          'Macro & Structure',
    'Industry (including construction), value added (% of GDP)': 'Macro & Structure',
    'Trade (% of GDP)':                          'Macro & Structure',
    'Government revenue':                        'Macro & Structure',
    'Inflation, consumer prices (annual %)':     'Macro & Structure',
    'Real interest rate (%)':                    'Macro & Structure',
    'Lending interest rate (%)':                 'Macro & Structure',
    'Domestic credit to private sector (% of GDP)': 'Finance & Investment',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Finance & Investment',
    'Adjusted savings: gross savings (% of GNI)': 'Finance & Investment',
    'Capital depreciation rate':                 'Finance & Investment',
    'Human capital index':                       'Human Capital & Infrastructure',
    'Life expectancy at birth, total (years)':   'Human Capital & Infrastructure',
    'Access to electricity (% of population)':   'Human Capital & Infrastructure',
    'Mobile cellular subscriptions (per 100 people)': 'Human Capital & Infrastructure',
    'Urban population (% of total population)':  'Human Capital & Infrastructure',
    'Death rate, crude (per 1,000 people)':      'Human Capital & Infrastructure',
    'Political stability \u2014 estimate':        'Governance',
    'Rule of law index':                         'Governance',
    'Property rights':                           'Governance',
    'Political corruption index':                'Governance',
}

CAT_COLORS = {
    'Resource Rents':                  '#E63946',
    'Macro & Structure':               '#9B59B6',
    'Finance & Investment':            '#E9C46A',
    'Human Capital & Infrastructure':  '#2A9D8F',
    'Governance':                      '#457B9D',
}

eci_col = 'Economic Complexity Index'
feat_cols = [c for c in CATEGORY_MAP.keys() if c in panel.columns]

corr_rows = []
for col in feat_cols:
    sub = panel[[eci_col, col]].dropna()
    if len(sub) < 20:
        continue
    r = sub[eci_col].corr(sub[col])
    corr_rows.append({
        'Feature': col, 'Correlation': r,
        'Category': CATEGORY_MAP.get(col, 'Other'),
        'Label': shorten(col),
    })

corr_df = pd.DataFrame(corr_rows).sort_values('Correlation', ascending=True).reset_index(drop=True)

fig02 = go.Figure()
for cat, color in CAT_COLORS.items():
    sub = corr_df[corr_df['Category'] == cat]
    if len(sub) == 0:
        continue
    fig02.add_trace(go.Bar(
        x=sub['Correlation'], y=sub['Label'], orientation='h',
        marker=dict(color=color, opacity=0.88, line=dict(color='white', width=0.5)),
        name=cat,
        hovertemplate='%{y}: r = %{x:.3f}<extra>' + cat + '</extra>',
    ))

fig02.add_vline(x=0, line=dict(color='#444', width=1.5))
fig02.update_layout(**base_layout(
    barmode='relative', height=700,
    margin=dict(l=200, r=80, t=60, b=60),
    xaxis=dict(title='Pearson r with ECI', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(tickfont=dict(size=10), categoryorder='array',
               categoryarray=corr_df['Label'].tolist()),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig02, '02_correlations_with_eci', w=1100, h=700)


  Saved: Final/charts/descriptive/02_correlations_with_eci.png


## Chart 26 — PCA resource loadings heatmap

In [4]:
nr_1995 = nr_sample[nr_sample['Year'] == 1995]

pivot = nr_1995.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Population'],
    columns='Resource', values='Production_TotalValue',
).reset_index()

res_cols = [c for c in pivot.columns
            if c not in ['Country', 'Country Code', 'Year', 'Population']]
pivot[res_cols] = pivot[res_cols].div(pivot['Population'], axis=0)
pivot = pivot.fillna(0)

X = np.log1p(pivot[res_cols].fillna(0))
pca = PCA(n_components=2, random_state=42)
pca.fit(X)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=res_cols)
top20 = loadings.abs().sum(axis=1).nlargest(20).index
plot_df = loadings.loc[top20]
plot_df = (plot_df.assign(_s=plot_df['PC1'].abs() + plot_df['PC2'].abs())
           .sort_values('_s', ascending=False).drop(columns='_s'))

y_labels = [
    f'PC2 ({var2:.1f}%) — Copper, Gold & Coal',
    f'PC1 ({var1:.1f}%) — Oil & Gas',
]

fig26 = go.Figure(go.Heatmap(
    z=plot_df[['PC2', 'PC1']].T.values,
    x=plot_df.index.tolist(), y=y_labels,
    colorscale=[[0.0, '#1a4a8a'], [0.5, '#ffffff'], [1.0, '#c23a3a']],
    zmid=0, zmin=-1, zmax=1,
    hovertemplate='<b>%{x}</b><br>%{y}: %{z:.3f}<extra></extra>',
    colorbar=dict(title='Loading', thickness=18, len=1.0,
                  tickvals=[-1, -0.5, 0, 0.5, 1]),
))
fig26.update_layout(**base_layout(
    height=420, margin=dict(l=260, r=120, t=60, b=160),
    xaxis=dict(tickangle=-40, tickfont=dict(size=10), showgrid=False),
    yaxis=dict(tickfont=dict(size=12), showgrid=False),
))
save(fig26, '26_pca_loadings_heatmap', w=1300, h=420)


  Saved: Final/charts/descriptive/26_pca_loadings_heatmap.png


## Chart 03 — PCA biplot (k=5, 1995)

In [5]:
cl_1995 = pd.read_csv('intermediary/clusters1995.csv')

# Recompute PCA for biplot arrows (need loadings)
loadings_biplot = pd.DataFrame(
    pca.components_.T * np.sqrt(pca.explained_variance_),
    columns=['PC1', 'PC2'], index=res_cols,
)
importance = loadings_biplot.abs().sum(axis=1)
top5  = importance.nlargest(5).index
top10 = importance.nlargest(10).index
scale = 2.8

fig03 = go.Figure()

for lbl in sorted(cl_1995['ClusterLabels'].unique()):
    sub = cl_1995[cl_1995['ClusterLabels'] == lbl]
    color = CLUSTER_COLORS.get(lbl, '#999')
    fig03.add_trace(go.Scatter(
        x=sub['PC1'], y=sub['PC2'], mode='markers+text',
        marker=dict(size=10, color=color, opacity=0.82,
                    line=dict(width=1.2, color='white')),
        text=sub['Country Code'], textposition='top center',
        textfont=dict(size=8, color='#333'), name=lbl,
        hovertemplate='<b>%{text}</b><br>PC1=%{x:.2f}, PC2=%{y:.2f}<extra></extra>',
    ))

# Biplot arrows
for feat in top10:
    is_top5 = feat in top5
    x1 = loadings_biplot.loc[feat, 'PC1'] * scale
    y1 = loadings_biplot.loc[feat, 'PC2'] * scale
    fig03.add_annotation(
        x=x1, y=y1, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True,
        arrowhead=3 if is_top5 else 2,
        arrowsize=1.2 if is_top5 else 0.8,
        arrowwidth=2.2 if is_top5 else 1.2,
        arrowcolor='#222' if is_top5 else 'rgba(150,150,150,0.5)',
    )
    if is_top5:
        fig03.add_annotation(
            x=x1 * 1.18, y=y1 * 1.18,
            text=f'<b>{feat}</b>', showarrow=False,
            font=dict(size=10, color='#111', family=FONT),
            bgcolor='rgba(255,255,255,0.7)', borderpad=2,
        )

fig03.add_hline(y=0, line=dict(color=GRID, width=1))
fig03.add_vline(x=0, line=dict(color=GRID, width=1))

fig03.update_layout(**base_layout(
    height=680, margin=dict(l=60, r=60, t=50, b=60),
    xaxis=dict(title=f'PC1 ({var1:.1f}% variance)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title=f'PC2 ({var2:.1f}% variance)', gridcolor=GRID, gridwidth=0.5),
    legend=dict(title='Resource profile (k=4, 1995)', font=dict(size=10),
                bgcolor='rgba(250,250,250,0.85)', bordercolor=GRID, borderwidth=1),
))
save(fig03, '03_pca_biplot_k5', w=1100, h=680)

  Saved: Final/charts/descriptive/03_pca_biplot_k5.png


## Chart 04 — Cluster world map (k=5, 1995)

In [6]:
def create_cluster_map(cl_df):
    fig = go.Figure()
    for lbl in sorted(cl_df['ClusterLabels'].unique()):
        sub   = cl_df[cl_df['ClusterLabels'] == lbl]
        color = CLUSTER_COLORS.get(lbl, '#aaa')
        fig.add_trace(go.Choropleth(
            locations=sub['Country Code'],
            z=[1] * len(sub),
            colorscale=[[0, color], [1, color]],
            showscale=False,
            showlegend=True,
            name=lbl,
            text=sub['Country'],
            hovertemplate='<b>%{text}</b><br>' + lbl + '<extra></extra>',
            marker=dict(line=dict(color='white', width=0.6)),
        ))

    fig.update_geos(
        projection_type='natural earth',
        showcountries=True, countrycolor='#d0d0d0',
        showcoastlines=False,
        showland=True, landcolor='#f0f0f0',
        showocean=True, oceancolor='#dde8f0',
        showframe=False,
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=30, b=60),
        legend=dict(
            orientation='h', x=0.5, y=-0.06,
            xanchor='center', yanchor='top',
            font=dict(size=11, family=FONT),
            bgcolor='rgba(250,250,250,0.9)',
            bordercolor='#d0d0d0', borderwidth=1,
        ),
        paper_bgcolor=BG, font=dict(family=FONT),
    )
    return fig

fig04 = create_cluster_map(cl_map)
fig04.update_layout(height=520)
save(fig04, '04_cluster_world_map_k5', w=1200, h=520)

  Saved: Final/charts/descriptive/04_cluster_world_map_k5.png


## Chart 05 — ECI vs GDP per capita (animated, 1995-2019)

In [7]:
cl_agg = pd.read_csv('intermediary/clustersagg.csv')
cl_agg_map = cl_agg[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')

data05 = panel.merge(cl_agg_map[['Country Code', 'Cluster', 'ClusterLabels']],
                     on='Country Code', how='left', suffixes=('', '_agg'))
# Use agg cluster labels if available
if 'ClusterLabels_agg' in data05.columns:
    data05['ClusterLabels'] = data05['ClusterLabels_agg'].fillna(data05['ClusterLabels'])
    data05['Cluster'] = data05['Cluster_agg'].fillna(data05['Cluster_x'] if 'Cluster_x' in data05.columns else data05['Cluster'])

data05['Log_GDP_pc'] = np.log(data05['GDP per capita (constant prices, PPP)'].replace(0, np.nan))
data05['Prod_pc'] = data05['Total_Production_Value'] / data05['Population'].replace(0, np.nan)
data05 = data05.dropna(subset=['Log_GDP_pc', 'Economic Complexity Index', 'ClusterLabels'])

# Bubble size
data05['Bubble'] = np.sqrt(data05['Prod_pc'].fillna(0))
mn, mx = data05['Bubble'].min(), data05['Bubble'].max()
data05['Bubble_Scaled'] = 8 + (data05['Bubble'] - mn) / (mx - mn + 1e-9) * 40

years = sorted(data05['Year'].unique())
countries = data05['Country Code'].unique()

# Build country data dict
cdata = {}
for cc in countries:
    cdf = data05[data05['Country Code'] == cc].sort_values('Year')
    origin = cdf[cdf['Year'] == 1995]
    if len(origin) == 0:
        continue
    cdata[cc] = {
        'years': cdf['Year'].values,
        'x': cdf['Log_GDP_pc'].values,
        'y': cdf['Economic Complexity Index'].values,
        'x0': origin['Log_GDP_pc'].values[0],
        'y0': origin['Economic Complexity Index'].values[0],
        'size': cdf['Bubble_Scaled'].values,
        'name': cdf['Country Name'].iloc[0],
        'cluster': cdf['ClusterLabels'].iloc[0],
    }

valid_cc = list(cdata.keys())

fig05 = go.Figure()

# Initial state (first year)
for lbl in sorted(CLUSTER_COLORS.keys()):
    cc_list = [c for c in valid_cc if cdata[c]['cluster'] == lbl]
    if not cc_list:
        continue
    color = CLUSTER_COLORS[lbl]

    for cc in cc_list:
        cd = cdata[cc]
        idx = np.where(cd['years'] == years[0])[0]
        if len(idx) > 0:
            i = idx[0]
            xv, yv, sv = [cd['x'][i]], [cd['y'][i]], cd['size'][i]
        else:
            xv, yv, sv = [cd['x0']], [cd['y0']], 15
        fig05.add_trace(go.Scatter(
            x=xv, y=yv, mode='markers+text',
            marker=dict(size=sv, color=color, opacity=0.85,
                        line=dict(width=1.5, color='white')),
            text=[cc], textposition='top center',
            textfont=dict(size=8, color='black'),
            name=lbl, legendgroup=lbl,
            showlegend=(cc == cc_list[0]),
            customdata=[[cd['name']]],
            hovertemplate='<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>ECI: %{y:.2f}<extra></extra>',
        ))

# Frames
frames = []
for year in years:
    fd = []
    for lbl in sorted(CLUSTER_COLORS.keys()):
        cc_list = [c for c in valid_cc if cdata[c]['cluster'] == lbl]
        color = CLUSTER_COLORS[lbl]
        for cc in cc_list:
            cd = cdata[cc]
            idx = np.where(cd['years'] == year)[0]
            if len(idx) > 0:
                i = idx[0]
                xv, yv, sv = [cd['x'][i]], [cd['y'][i]], cd['size'][i]
            else:
                mask = cd['years'] <= year
                if mask.any():
                    li = np.where(mask)[0][-1]
                    xv, yv, sv = [cd['x'][li]], [cd['y'][li]], cd['size'][li]
                else:
                    xv, yv, sv = [cd['x0']], [cd['y0']], 15
            fd.append(go.Scatter(
                x=xv, y=yv, mode='markers+text',
                marker=dict(size=sv, color=color, opacity=0.85,
                            line=dict(width=1.5, color='white')),
                text=[cc], textposition='top center', textfont=dict(size=8),
                customdata=[[cd['name']]],
                hovertemplate='<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>ECI: %{y:.2f}<extra></extra>',
            ))
    frames.append(go.Frame(data=fd, name=str(year)))

fig05.frames = frames

eci_all = data05['Economic Complexity Index']
x_all = data05['Log_GDP_pc']
fig05.update_layout(
    xaxis=dict(range=[x_all.min()-0.2, x_all.max()+0.2],
               title='Log GDP per capita (PPP, constant 2017 USD)'),
    yaxis=dict(range=[eci_all.min()-0.5, eci_all.max()+0.5],
               title='Economic Complexity Index'),
    plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(family=FONT, color=NAVY),
    legend=dict(title='Resource profile (k=5)', x=1.02, y=0.99),
    updatemenus=[dict(
        type='buttons', showactive=True, x=1.0, y=-0.02,
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, dict(frame=dict(duration=500, redraw=True),
                                  transition=dict(duration=300))]),
            dict(label='Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0), mode='immediate')]),
        ],
    )],
    sliders=[dict(
        active=0, len=0.85, x=0.05, y=-0.12,
        currentvalue=dict(prefix='Year: ', font=dict(size=14)),
        steps=[dict(
            args=[[str(y)], dict(frame=dict(duration=300, redraw=True), mode='immediate')],
            method='animate', label=str(y),
        ) for y in years],
    )],
)
save(fig05, '05_eci_vs_gdp_animated_k5', w=1200, h=700)


  Saved: Final/charts/descriptive/05_eci_vs_gdp_animated_k5.png


## Chart 06 — Cluster profile comparison (grouped bars)

In [8]:
# Grouped bar chart: normalised means per cluster across key variables
PROFILE_VARS = {
    'Total natural resources rents (% of GDP)': 'NR Rents',
    'Oil rents (% of GDP)':                      'Oil Rents',
    'GDP per capita (constant prices, PPP)':     'GDP per capita',
    'Agriculture, forestry, and fishing, value added (% of GDP)': 'Agriculture',
    'Domestic credit to private sector (% of GDP)': 'Domestic Credit',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'GFCF',
    'Human capital index':                       'Human Capital',
    'Life expectancy at birth, total (years)':   'Life Expectancy',
    'Rule of law index':                         'Rule of Law',
    'Political corruption index':                'Pol. Corruption',
}

# Compute cluster means, then normalise 0-1 within each variable
available = {k: v for k, v in PROFILE_VARS.items() if k in panel.columns}
cluster_means = panel.groupby('ClusterLabels')[list(available.keys())].mean()

# Min-max normalise — .where() prevents the division ever touching zero-range columns
_lo  = cluster_means.min()
_rng = cluster_means.max() - _lo          # range per variable
_safe = _rng.where(_rng > 0)              # NaN where range == 0; no division happens
normed = cluster_means.sub(_lo).div(_safe).fillna(0.5)

normed = normed.rename(columns=available)

fig06 = go.Figure()
for lbl in sorted(normed.index):
    color = CLUSTER_COLORS.get(lbl, '#999')
    fig06.add_trace(go.Bar(
        x=normed.columns.tolist(),
        y=normed.loc[lbl].values,
        name=lbl,
        marker=dict(color=color, opacity=0.88),
        hovertemplate='%{x}: %{y:.2f}<extra>' + lbl + '</extra>',
    ))

fig06.update_layout(**base_layout(
    barmode='group', height=500,
    margin=dict(l=60, r=40, t=60, b=120),
    xaxis=dict(tickangle=-30, tickfont=dict(size=10)),
    yaxis=dict(title='Normalised mean (0 = sample min, 1 = sample max)',
               gridcolor=GRID, gridwidth=0.5, range=[0, 1.05]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig06, '06_cluster_profile_comparison_k5', w=1200, h=500)


  Saved: Final/charts/descriptive/06_cluster_profile_comparison_k5.png


## Chart 14 — ECI distribution shift, 1995 vs 2019

In [9]:
yr_95 = panel[panel['Year'] == 1995]['Economic Complexity Index'].dropna().sort_values().values
yr_19 = panel[panel['Year'] == 2019]['Economic Complexity Index'].dropna().sort_values().values

fig14 = go.Figure()
for vals, yr, col in [(yr_95, 1995, '#457B9D'), (yr_19, 2019, '#E63946')]:
    pcts = np.linspace(0, 100, len(vals))
    fig14.add_trace(go.Scatter(
        x=pcts, y=vals, mode='lines', name=str(yr),
        line=dict(color=col, width=2.5),
        hovertemplate=f'{yr} | P%{{x:.0f}}: %{{y:.3f}}<extra></extra>',
    ))
fig14.update_layout(**base_layout(
    height=500,
    xaxis=dict(title='Percentile', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=11)),
))
save(fig14, '14_eci_distribution_shift', w=1100, h=500)


  Saved: Final/charts/descriptive/14_eci_distribution_shift.png


## Chart 15 — ECI median trajectory by cluster

In [10]:
traj = panel.groupby(['Year', 'ClusterLabels'])['Economic Complexity Index'].median().reset_index()

fig15 = go.Figure()
for lbl in sorted(traj['ClusterLabels'].dropna().unique()):
    sub = traj[traj['ClusterLabels'] == lbl]
    fig15.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers', name=lbl,
        line=dict(color=CLUSTER_COLORS.get(lbl, '#999'), width=2.2),
        marker=dict(size=5),
        hovertemplate='%{x}: %{y:.3f}<extra>' + lbl + '</extra>',
    ))
fig15.update_layout(**base_layout(
    height=480,
    xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5, dtick=5),
    yaxis=dict(title='Median ECI', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
    hovermode='x unified',
))
save(fig15, '15_eci_trajectory_by_cluster_k5', w=1100, h=480)


  Saved: Final/charts/descriptive/15_eci_trajectory_by_cluster_k5.png


## Charts 16–18 — ECI Trajectories: Case Study Comparisons

In [11]:

TRAJ_NAMES = {
    'AZE': 'Azerbaijan', 'UZB': 'Uzbekistan', 'MYS': 'Malaysia', 'IRQ': 'Iraq',
    'COG': 'Congo',      'AGO': 'Angola',      'CHL': 'Chile',
    'BWA': 'Botswana',   'ZMB': 'Zambia',      'NOR': 'Norway',
}

CHARTS = [
    {
        'fname': '16_eci_traj_congo_comparators',
        'title': 'ECI Trajectory: Congo vs Comparators',
        'groups': [
            ('COG', '#c23a3a'),
            ('MYS', '#2e7d4a'),
            ('AZE', '#4a6fa5'),
            ('AGO', '#c97030'),
        ],
    },
    {
        'fname': '17_eci_traj_az_comparators',
        'title': 'ECI Trajectory: Azerbaijan vs Comparators',
        'groups': [
            ('AZE', '#4a6fa5'),
            ('MYS', '#2e7d4a'),
            ('UZB', '#c97030'),
            ('IRQ', '#8B5CF6'),
        ],
    },
    {
        'fname': '18_eci_traj_chile_comparators',
        'title': 'ECI Trajectory: Chile vs Comparators',
        'groups': [
            ('CHL', '#c23a3a'),
            ('BWA', '#4a6fa5'),
            ('ZMB', '#2e7d4a'),
            ('NOR', '#333333'),
        ],
    },
]

eci_full = master[['Country Code', 'Year', 'Economic Complexity Index']].dropna()

for chart in CHARTS:
    fig = go.Figure()
    for cc, col in chart['groups']:
        sub = eci_full[eci_full['Country Code'] == cc].sort_values('Year')
        name = TRAJ_NAMES.get(cc, cc)
        fig.add_trace(go.Scatter(
            x=sub['Year'], y=sub['Economic Complexity Index'],
            mode='lines+markers', name=name,
            line=dict(color=col, width=2.2),
            marker=dict(size=5, color=col),
            hovertemplate=f'<b>{name}</b>  %{{x}}: %{{y:.3f}}<extra></extra>',
        ))
    fig.update_layout(**base_layout(
        height=480,
        title=dict(text=chart['title'], x=0.02, font=dict(size=13, color=NAVY)),
        xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5, dtick=5, range=[1994, 2020]),
        yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
                   zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
        legend=dict(font=dict(size=11), bgcolor='rgba(255,255,255,0.9)',
                    bordercolor=GRID, borderwidth=1),
        hovermode='x unified',
        margin=dict(l=70, r=160, t=55, b=55),
    ))
    save(fig, chart['fname'], w=1100, h=480)


  Saved: Final/charts/descriptive/16_eci_traj_congo_comparators.png
  Saved: Final/charts/descriptive/17_eci_traj_az_comparators.png
  Saved: Final/charts/descriptive/18_eci_traj_chile_comparators.png
